### In this experiment, I used a LightGBM model with the best hyperparameters obtained from the previous LightGBM optimization. LightGBM was chosen because it trains significantly faster while achieving slightly better performance than XGBoost.


In [1]:
!pip install mlflow 
!pip install boto3
!pip install awscli

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 85.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 85.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.1/969.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [14]:
import os
import mlflow
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow.lightgbm
import lightgbm as lgb
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [3]:
data = pd.read_csv("/kaggle/input/datasets/bjdhdhjdbd/twitter-data-sentiment/clean_data.csv")
data = data.dropna()
data.head()

,clean_text,category
0,family mormon never try explain still stare pu...,1.0
1,buddhism much lot compatible christianity espe...,1.0
2,seriously say thing first get complex explain ...,-1.0
3,learn want teach different focus goal not wrap...,0.0
4,benefit may want read live buddha live christ ...,1.0


In [4]:
vectorizer = CountVectorizer(max_features=1000)

In [5]:

X = vectorizer.fit_transform(data['clean_text']).toarray()
y = data["category"]

In [8]:
x_train , x_test, y_train  , y_test = train_test_split(X , y , test_size=0.2, random_state=42)
mapping = {
    -1: 0,
     0: 1,
     1: 2
}
y_train = y_train.map(mapping)
y_test = y_test.map(mapping)

In [13]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

print(class_weight_dict)

{np.int64(0): np.float64(1.5160720392112161), np.int64(1): np.float64(0.9826563971851278), np.int64(2): np.float64(0.7560001705199437)}


In [27]:
lgbm_params = {
    "colsample_bytree": 0.8997767208035865,
    "learning_rate": 0.13702846406786778,
    "max_depth": 7,
    "min_child_samples": 73,
    "n_estimators": 733,
    "num_leaves": 149,
    "reg_alpha": 2.1208903623034105,
    "reg_lambda": 3.2514205087388133,
    "subsample": 0.6765419227639857,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1
}

In [28]:
approach_list = ["Class_Weights","SMOTE", "UnderSampling" , "ADASYN", "Oversampling"]

def augmentation(X_train, y_train, approach):

    if approach == "Class_Weights":

        return X_train, y_train, class_weight_dict

    elif approach == "SMOTE":
        sampler = SMOTE(random_state=42)
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        return X_res, y_res, None

    elif approach == "ADASYN":
        sampler = ADASYN(random_state=42)
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        return X_res, y_res, None

    elif approach == "Oversampling":
        sampler = RandomOverSampler(random_state=42)
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        return X_res, y_res, None

    elif approach == "UnderSampling":
        sampler = RandomUnderSampler(random_state=42)
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        return X_res, y_res, None

    else:
        raise ValueError(f"Unknown approach: {approach}")
    

In [29]:
def evaluation(model , approach , y_test):
    y_pred = model.predict(x_test)
    rev_mapping = {
    0: -1,
    1: 0,
    2: 1
    }

    y_pred = np.vectorize(rev_mapping.get)(y_pred)
    y_test = np.vectorize(rev_mapping.get)(y_test)
    
    classification_rep = classification_report(
    y_test,
    y_pred,
    output_dict=True
    )
    print("#"*25 , f"{approach}","#"*25 )
    print(f"{classification_rep} ")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy:.4f}")
    
    return accuracy, classification_rep ,  y_pred , y_test

In [30]:
def log_mlflow(approach , model , accuracy , classification_rep ,  y_pred , y_test): 
    
 try:
    with mlflow.start_run() as run:


        mlflow.set_tag("model", "LGBM")
        mlflow.set_tag("description", f"Model trained using {approach}")
        mlflow.log_param("vectorizer_type", "CountVectorizer")
        mlflow.log_param("vectorizer_max_features", 1000)
        mlflow.log_params(lgbm_params)
        mlflow.log_param("augmentation", approach)

        mlflow.log_metric("accuracy", accuracy)

        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", value)


        mlflow.lightgbm.log_model(
            lgb_model=model,
           artifact_path="lightgbm_model"
           )

        data.to_csv("dataset.csv", index=False)
        mlflow.log_artifact("dataset.csv")

        conf_matrix = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("Confusion Matrix")

        plt.savefig("confusion_matrix.png")
        plt.close()

        mlflow.log_artifact("confusion_matrix.png")

        print(f"Run ID: {run.info.run_id}")

 except Exception as e:
    print(f"MLflow logging failed: {e}")

 finally:
    
    import os

    if os.path.exists("dataset.csv"):
        os.remove("dataset.csv")

    if os.path.exists("confusion_matrix.png"):
        os.remove("confusion_matrix.png")

In [31]:
def train_model(approach):
 X_train_aug , y_train_aug , class_weights  = augmentation(x_train, y_train, approach)
 if approach == "Class_Weights":
    model = lgb.LGBMClassifier(
        **lgbm_params,
        class_weight=class_weights
    )
 else:
    model = lgb.LGBMClassifier(
        **lgbm_params
    )
 
 model.fit(X_train_aug, y_train_aug)
    
 accuracy, classification_rep ,  y_pred , y_test_decoded = evaluation(model , approach , y_test)
 log_mlflow(approach , model , accuracy , classification_rep ,  y_pred , y_test_decoded)
 

In [33]:
mlflow.set_tracking_uri("http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/")
mlflow.set_experiment("LGBM_Model_Balanced_Dataset ")

2026/07/26 19:56:16 INFO mlflow.tracking.fluent: Experiment with name 'LGBM_Model_Balanced_Dataset ' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://zg-mlflow/3', creation_time=1785095776198, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1785095776198, lifecycle_stage='active', name='LGBM_Model_Balanced_Dataset ', tags={}, trace_location=None, workspace='default'>

In [34]:
for approach in approach_list: 
    train_model(approach)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


######################### Class_Weights #########################
{'-1': {'precision': 0.7109720176730486, 'recall': 0.6689376443418014, 'f1-score': 0.6893146120894812, 'support': 8660.0}, '0': {'precision': 0.7300856962098119, 'recall': 0.9313962873284907, 'f1-score': 0.8185452669589889, 'support': 13629.0}, '1': {'precision': 0.9031112967216538, 'recall': 0.7366717765287004, 'f1-score': 0.8114446529080676, 'support': 17613.0}, 'accuracy': 0.7884817803618867, 'macro avg': {'precision': 0.7813896702015048, 'recall': 0.7790019027329974, 'f1-score': 0.7731015106521792, 'support': 39902.0}, 'weighted avg': {'precision': 0.8023120368866377, 'recall': 0.7884817803618867, 'f1-score': 0.7873638578454403, 'support': 39902.0}} 
Accuracy: 0.7885


2026/07/26 19:57:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: 53ca8bf8a44e438aa453b5e2e84f33d6
🏃 View run vaunted-toad-369 at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/3/runs/53ca8bf8a44e438aa453b5e2e84f33d6
🧪 View experiment at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/3


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

Run ID: 3390ae8a8f094471b305cdaec2274c87
🏃 View run agreeable-doe-499 at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/3/runs/3390ae8a8f094471b305cdaec2274c87
🧪 View experiment at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/3


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


######################### Oversampling #########################
{'-1': {'precision': 0.7126223213179735, 'recall': 0.6643187066974596, 'f1-score': 0.6876232594274786, 'support': 8660.0}, '0': {'precision': 0.7301267281105991, 'recall': 0.9300022011886419, 'f1-score': 0.8180322049759592, 'support': 13629.0}, '1': {'precision': 0.8991637293524086, 'recall': 0.7386589450973713, 'f1-score': 0.811046692849573, 'support': 17613.0}, 'accuracy': 0.7878803067515413, 'macro avg': {'precision': 0.7806375929269938, 'recall': 0.777659950994491, 'f1-score': 0.7722340524176703, 'support': 39902.0}, 'weighted avg': {'precision': 0.8009417383869726, 'recall': 0.7878803067515413, 'f1-score': 0.7866458761319944, 'support': 39902.0}} 
Accuracy: 0.7879


2026/07/26 20:16:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: b8510f4787794de6849d2eef15741573
🏃 View run fearless-pig-42 at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/3/runs/b8510f4787794de6849d2eef15741573
🧪 View experiment at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/3
